<a href="https://colab.research.google.com/github/LGF92/italian-g2p/blob/main/italian-g2p.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/CUNY-CL/wikipron/master/data/scrape/tsv/ita_latn_broad.tsv"
lex = pd.read_csv(url, sep="\t", names=["word", "ipa"])
print(len(lex))
lex.sample(10, random_state=42)

89608


,word,ipa
17150,budelli,b u d ɛ l l i
49238,manoscritto,m a n o s k r i t t o
35952,formali,f o r m a l i
43667,innestati,i n n ɛ s t a t i
34614,fendere,f ɛ n d e r e
44612,intervistare,i n t e r v i s t a r e
81733,talebano,t a l e b a n o
26394,degnare,d e ɲ ɲ a r e
3300,Picerno,p i t͡ʃ ɛ r n o
31526,embrione,e m b r i o n e


In [2]:
import re
import unicodedata

# ---------------------------------------------------------------
# 1. Spelling-to-sound rules (regex pattern, IPA output)
#    Order matters: longer / more specific spellings come first,
#    because the first rule that matches wins.
#    (?=...) is a lookahead: it checks the next letter without consuming it.
# ---------------------------------------------------------------
V = "aeiouɛɔ"  # all vowels, including open ɛ and ɔ

RULES = [
    (rf"sci(?=[{V}])", "ʃ"),       # sciarpa, scienza: i is silent before a vowel
    (r"sc(?=[eiɛ])", "ʃ"),         # pesce, sci: sc before e/i
    (rf"gli(?=[{V}])", "ʎ"),       # figlio: i is silent before a vowel
    (r"gli", "ʎ i"),               # gli, figli: i is pronounced
    (r"gn", "ɲ"),                  # bagno
    (r"ch(?=[eiɛ])", "k"),         # che, chiesa: hard c
    (r"gh(?=[eiɛ])", "ɡ"),         # ghiro: hard g
    (rf"ci(?=[{V}])", "tʃ"),       # ciao, cielo: i is silent before a vowel
    (r"c(?=[eiɛ])", "tʃ"),         # cena, cibo
    (rf"gi(?=[{V}])", "dʒ"),       # giallo, Gianna: i is silent before a vowel
    (r"g(?=[eiɛ])", "dʒ"),         # gelato, giro
    (r"qu", "k w"),                # quale
    (r"c", "k"),                   # casa
    (r"g", "ɡ"),                   # gatto (IPA ɡ, not the keyboard g)
    (r"z", "ts"),                  # default voiceless; voiced dz handled in fix()
    (r"x", "k s"),                 # taxi
    (r"y", "i"),                   # loanwords
    (r"h", ""),                    # h is silent
    (rf"i(?=[{V}])", "j"),         # piede: i before a vowel is a glide
    (rf"u(?=[{V}])", "w"),         # uomo: u before a vowel is a glide
]

# ---------------------------------------------------------------
# 2. Grapheme-to-phoneme: spelling in, IPA out
# ---------------------------------------------------------------
def g2p(word):
    # Lowercase, and keep vowel quality from grave accents: è = open ɛ, ò = open ɔ
    w = word.lower().replace("è", "ɛ").replace("ò", "ɔ")
    # Split other accented letters into letter + accent mark, then drop the marks
    w = unicodedata.normalize("NFKD", w)
    w = "".join(ch for ch in w if not unicodedata.combining(ch))

    out, i = [], 0
    while i < len(w):
        for pat, sound in RULES:
            m = re.match(pat, w[i:])
            if m:                      # first matching rule wins
                out.append(sound)
                i += m.end()           # skip the letters we just used
                break
        else:                          # no rule matched: the letter is its own sound
            out.append(w[i])
            i += 1

    # Join into space-separated phonemes (empty strings from h disappear)
    s = " ".join(" ".join(out).split())

    # Inherently long consonants: ʃ ɲ ʎ ts dz are doubled between vowels (pesce, bagno)
    s = re.sub(rf"(?<=[{V}] )(ʃ|ɲ|ʎ|ts|dz) (?=[{V}jw])", r"\1 \1 ", s)

    # Write doubled affricates the way Wiktionary does: cc = t tʃ, gg = d dʒ, zz = t ts
    s = s.replace("k tʃ", "t tʃ").replace("ɡ dʒ", "d dʒ")
    s = s.replace("ts ts", "t ts").replace("dz dz", "d dz")

    # s becomes voiced z between vowels and before voiced consonants (casa, sbaglio)
    s = re.sub(rf"(?<=[{V}] )s(?= [{V}])", "z", s)
    s = re.sub(r"(?<= )s(?= [bdɡlmnrv])", "z", s)
    s = re.sub(r"^s(?= [bdɡlmnrv])", "z", s)
    return s

# ---------------------------------------------------------------
# 3. Fixes that need the whole word (endings, morphology)
# ---------------------------------------------------------------
# Endings where the i is stressed (fiaschetterìa, psicologìa): full vowel, not glide
HIATUS_END = ("eria", "erie", "logia", "logie", "grafia", "grafie",
              "fobia", "filia", "mania", "patia")

def fix(word, s):
    w = word.lower()
    # -oso/-osa/-osi/-ose: standard Italian keeps a voiceless s (accidioso)
    if re.search(r"os[aeio]$", w):
        s = re.sub(r"z (?=[aeio]$)", "s ", s)
    # Stressed -ìa endings: hiatus, not glide
    if w.endswith(HIATUS_END):
        s = re.sub(r"j (?=[ae]$)", "i ", s)
    # -ual/-uar/-uos: u stays a vowel (eventuale), except after k/ɡ (quale, guardare)
    s = re.sub(r"(?<![kɡ] )w (?=(a l|a r|o s|ɔ s))", "u ", s)
    # -izz-: voiced dz (terrorizzare); first occurrence only
    if "izz" in w:
        s = s.replace("i t ts", "i d dz", 1)
    return s

# ---------------------------------------------------------------
# 4. Score against Wiktionary
# ---------------------------------------------------------------
# Clean the reference: remove primary/secondary stress marks and tie bars
lex["clean"] = (lex["ipa"].str.replace("ˈ", "").str.replace("ˌ", "")
                .str.replace("\u0361", ""))

# Run g2p, then patch with fix()
lex["pred"] = lex["word"].apply(lambda w: fix(w, g2p(w)))
lex["ok"] = lex["pred"] == lex["clean"]

# Second score that treats open and closed vowels as the same
def merge(s):
    return s.replace("ɛ", "e").replace("ɔ", "o")

lex["ok_novowel"] = lex["pred"].apply(merge) == lex["clean"].apply(merge)

print(f"Exact match: {lex['ok'].mean():.1%}")
print(f"Ignoring open/closed vowels: {lex['ok_novowel'].mean():.1%}")
print(lex[~lex["ok_novowel"]].sample(10, random_state=1)[["word", "clean", "pred"]])

Exact match: 65.0%
Ignoring open/closed vowels: 86.5%
                  word                          clean  \
27365       diabetiche              d i a b ɛ t i k e   
78622            spesa                      s p e s a   
88896        zancleane             dz a n k l e a n e   
73654  scrupolosamente  s k r u p o l o s a m e n t e   
20783           cinesi                   tʃ i n e s i   
11449      areonautica          a r e o n a w t i k a   
55718       ortodossia            ɔ r t o d o s s i a   
19911           cercai                   tʃ e r k a j   
66548     riazzuffarsi       r i a d dz u f f a r s i   
849               Bush                          b u ʃ   

                                pred  
27365              d j a b e t i k e  
78622                      s p e z a  
88896             ts a n k l e a n e  
73654  s k r u p o l o z a m e n t e  
20783                   tʃ i n e z i  
11449          a r e o n a u t i k a  
55718            o r t o d o s s j a  
19911 

In [3]:
def merge(s):
    return s.replace("ɛ", "e").replace("ɔ", "o")

lex["ok_novowel"] = lex["pred"].apply(merge) == lex["clean"].apply(merge)
print(f"Exact match: {lex['ok'].mean():.1%}")
print(f"Ignoring open/closed vowels: {lex['ok_novowel'].mean():.1%}")
print(lex[~lex["ok_novowel"]].sample(10, random_state=1)[["word", "clean", "pred"]])

Exact match: 65.0%
Ignoring open/closed vowels: 86.5%
                  word                          clean  \
27365       diabetiche              d i a b ɛ t i k e   
78622            spesa                      s p e s a   
88896        zancleane             dz a n k l e a n e   
73654  scrupolosamente  s k r u p o l o s a m e n t e   
20783           cinesi                   tʃ i n e s i   
11449      areonautica          a r e o n a w t i k a   
55718       ortodossia            ɔ r t o d o s s i a   
19911           cercai                   tʃ e r k a j   
66548     riazzuffarsi       r i a d dz u f f a r s i   
849               Bush                          b u ʃ   

                                pred  
27365              d j a b e t i k e  
78622                      s p e z a  
88896             ts a n k l e a n e  
73654  s k r u p o l o z a m e n t e  
20783                   tʃ i n e z i  
11449          a r e o n a u t i k a  
55718            o r t o d o s s j a  
19911 

In [4]:
def fix2(word, s):
    s = fix(word, s)
    # Falling diphthongs: vowel + i/u + consonant or end of word -> glide (cercai, nautica)
    s = re.sub(rf"(?<=[{V}] )i(?=( [^{V} ])|$)", "j", s)
    s = re.sub(rf"(?<=[{V}] )u(?=( [^{V} ])|$)", "w", s)
    # -ese/-esi/-esa and -osamente: voiceless s in the standard
    if re.search(r"es[aeio]$|osamente$", word.lower()):
        s = re.sub(r"z(?= [aeio]( m e n t e)?$)", "s", s)
    return s

def score(pred_col):
    ok = lex[pred_col] == lex["clean"]
    ok_nv = lex[pred_col].apply(merge) == lex["clean"].apply(merge)
    return f"{ok.mean():.1%} exact, {ok_nv.mean():.1%} ignoring vowels"

# A: with the new fixes
lex["pred_a"] = lex["word"].apply(lambda w: fix2(w, g2p(w)))
print("With fixes:", score("pred_a"))

# Tested removing the s -> z rule between vowels: worse (63.8% vs 65.4% exact), so we keep it.

With fixes: 65.4% exact, 87.2% ignoring vowels


In [5]:
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

# Learn on 80% of words, test on the unseen 20% (no cheating)
train, test = train_test_split(lex, test_size=0.2, random_state=42)

def learn_endings(df, nmax=6, min_count=5, min_share=0.6):
    """For each ending of our output (last n phonemes), find how Wiktionary usually writes it."""
    table = {}
    for n in range(2, nmax + 1):
        counts = defaultdict(Counter)
        for p, c in zip(df["pred_a"], df["clean"]):
            pp, cc = p.split(), c.split()
            if len(pp) != len(cc) or len(pp) < n:   # only align words with the same length
                continue
            counts[" ".join(pp[-n:])][" ".join(cc[-n:])] += 1
        for end, cnt in counts.items():
            best, freq = cnt.most_common(1)[0]
            # keep it only if it's frequent and clearly the majority
            if freq >= min_count and freq / sum(cnt.values()) >= min_share:
                table[end] = best
    return table

def apply_endings(s, table, nmax=6):
    """Replace the longest known ending with its learned version."""
    ph = s.split()
    for n in range(min(nmax, len(ph)), 1, -1):
        end = " ".join(ph[-n:])
        if end in table:
            return " ".join(ph[:-n] + table[end].split())
    return s

table = learn_endings(train)
print("Endings learned:", len(table))

def score_df(df, col):
    ok = (df[col] == df["clean"]).mean()
    ok_nv = (df[col].apply(merge) == df["clean"].apply(merge)).mean()
    return f"{ok:.1%} exact, {ok_nv:.1%} ignoring vowels"

test = test.copy()
test["pred_v"] = test["pred_a"].apply(lambda s: apply_endings(s, table))
print("Test, rules only:    ", score_df(test, "pred_a"))
print("Test, + endings:     ", score_df(test, "pred_v"))

Endings learned: 6854
Test, rules only:     65.0% exact, 87.2% ignoring vowels
Test, + endings:      76.4% exact, 87.8% ignoring vowels


In [6]:
from collections import Counter

errs = test[test["pred_v"] != test["clean"]].copy()

def category(row):
    p, c = row["pred_v"], row["clean"]
    if merge(p) == merge(c):
        return "vowel quality only (e/ɛ, o/ɔ)"
    if len(p.split()) != len(c.split()):
        return "different length (glide/hiatus, doubling, foreign)"
    return "consonant or other swap"

errs["type"] = errs.apply(category, axis=1)
print(f"Test errors: {len(errs)} of {len(test)}\n")
print(errs["type"].value_counts(normalize=True).map("{:.1%}".format), "\n")

# Most common sound swaps, for words of the same length
swaps = Counter()
for p, c in zip(errs["pred_v"], errs["clean"]):
    pp, cc = p.split(), c.split()
    if len(pp) == len(cc):
        swaps.update(f"{a} → {b}" for a, b in zip(pp, cc) if a != b)
print("Top swaps (ours → Wiktionary):")
for s, n in swaps.most_common(10):
    print(f"  {s}: {n}")

Test errors: 4229 of 17922

type
vowel quality only (e/ɛ, o/ɔ)                         48.3%
consonant or other swap                               45.4%
different length (glide/hiatus, doubling, foreign)     6.4%
Name: proportion, dtype: object 

Top swaps (ours → Wiktionary):
  e → ɛ: 1240
  o → ɔ: 846
  j → i: 642
  z → s: 362
  s → z: 318
  ts → dz: 284
  ɛ → e: 224
  w → u: 143
  t → d: 127
  ɔ → o: 126


In [10]:
STRESS_V = set("aeiouɛɔ")   # vowels (glides j/w don't count as syllables)

def stress_pos(ipa):
    """Where the stress is, counted from the end:
    0 = last syllable (città), 1 = second-to-last (casa), 2 = third-to-last (tavolo)."""
    toks = ipa.replace("\u0361", "").split()
    n_vowels, stressed, pending = 0, None, False
    for t in toks:
        if "ˈ" in t:                      # stress mark: the next vowel is stressed
            pending = True
        base = t.replace("ˈ", "").replace("ˌ", "")
        if base in STRESS_V:
            if pending and stressed is None:
                stressed = n_vowels
                pending = False
            n_vowels += 1
    if stressed is None:                  # no stress mark (e.g. some monosyllables)
        return None
    return n_vowels - 1 - stressed

def baseline_stress(row):
    """Default Italian rule: second-to-last syllable, unless the word has one vowel
    or ends in a written accent (città, perché)."""
    w = row["word"].lower()
    n_vowels = sum(t in STRESS_V for t in row["pred_a"].split())
    if n_vowels <= 1 or w[-1] in "àèéìíòóùú":
        return 0
    return 1

# STRESS (proxy). This WikiPron file has no stress marks, so we can't learn stress directly.
# Proxy: in standard Italian, ɛ and ɔ only occur in the stressed syllable,
# so words with exactly one ɛ/ɔ tell us where the stress is.
# Better alternative for future work: the Kaikki.org Wiktionary extract,
# whose IPA (as far as we know) keeps the stress marks (ˈ), which would let us learn stress on all words.

def open_vowel_stress(clean):
    """Stress position counted from the end: 0 = last, 1 = second-to-last, 2 = third-to-last."""
    vowels = [t for t in clean.split() if t in STRESS_V]
    opens = [i for i, v in enumerate(vowels) if v in "ɛɔ"]
    if len(opens) != 1:
        return None
    return len(vowels) - 1 - opens[0]

test["stress"] = test["clean"].apply(open_vowel_stress)
t = test.dropna(subset=["stress"]).copy()
print(f"Test words with known stress (single ɛ/ɔ): {len(t)} of {len(test)}\n")

print("Where the stress falls:")
print(t["stress"].value_counts(normalize=True).sort_index().map("{:.1%}".format), "\n")

# Default rule: second-to-last syllable
t["stress_base"] = [baseline_stress(r) for _, r in t.iterrows()]
print(f"Default rule accuracy: {(t['stress_base'] == t['stress']).mean():.1%}")


Test words with known stress (single ɛ/ɔ): 4538 of 17922

Where the stress falls:
stress
0.0     3.2%
1.0    54.8%
2.0    34.2%
3.0     3.0%
4.0     2.5%
5.0     1.4%
6.0     0.7%
7.0     0.1%
8.0     0.1%
Name: proportion, dtype: object 

Default rule accuracy: 56.7%
